In [1]:
import pandas as pd
import os
import pickle
import bmra_prep
import bmra_prep.pathway_activity.prediction

In [2]:
cell_line ='BC3C_dec'

data_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}/00_outputs_2020_{cell_line}/"
out_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}/01_outputs_2020_{cell_line}/"


os.makedirs(out_dir, exist_ok = True)

# Load Data

In [3]:
# load metdadata dict and extract used elements
with open(os.path.join(data_dir, "metadata.pickle"), "rb") as f:
    all_metadata = pickle.load(f)

n_modules = all_metadata["n_modules"]
n_genes = all_metadata["n_genes"]
n_experiments = all_metadata["n_experiments"]

modules = all_metadata["modules"]
exp_ids = all_metadata["exp_ids"]
genes = all_metadata["genes"]

In [4]:
# load data
L1000_df = pd.read_csv(
    os.path.join(data_dir, "L1000_Data_norm_data.csv"),
    index_col = 0,
)

x = L1000_df.values
x.shape

(978, 106)

In [5]:
# load doses and perturbation matrix
inhib_conc_matrix = pd.read_csv(
    os.path.join(data_dir, "inhib_conc_annotated.csv"),
    index_col = 0,
).values

ic50_matrix = pd.read_csv(
    os.path.join(data_dir, "ic50_annotated.csv"),
    index_col = 0,
).values

# gamma_matrix = pd.read_csv(
#     os.path.join(data_dir, "gamma_annotated.csv"),
#     index_col = 0,
# ).values

pert_matrix = pd.read_csv(
    os.path.join(data_dir, "pert_annotated.csv"),
    index_col = 0,
).values

In [6]:
# y_true = (1 + gamma_matrix * inhib_conc_matrix / ic50_matrix) / (1 + inhib_conc_matrix / ic50_matrix)

y_true = 1 / (1 + inhib_conc_matrix / ic50_matrix)

display(y_true.shape)
y_true

(10, 106)

array([[1. , 1. , 1. , ..., 1. , 1. , 1. ],
       [1. , 1. , 1. , ..., 1. , 1. , 1. ],
       [1. , 1. , 1. , ..., 1. , 1. , 1. ],
       ...,
       [1. , 1. , 1. , ..., 1. , 1. , 1. ],
       [1. , 1. , 1. , ..., 1. , 1. , 1. ],
       [1. , 1. , 1. , ..., 0.5, 0.5, 0.5]])

## Run models

In [7]:
a_coeffs = bmra_prep.pathway_activity.prediction.predict_coeffs(
    x, y_true, pert_matrix, 200_000, 10, 10, 10, 100)

In [8]:
a_coeffs_df = pd.DataFrame(a_coeffs, index = modules, columns = genes)
a_coeffs_df.to_csv(os.path.join(out_dir, "a_coeffs.csv"))
#a_coeffs_df = pd.read_csv(os.path.join(out_dir,'a_coeffs.csv'),index_col=0)
#a_coeffs = a_coeffs_df.values
trh = 0.0001
display((abs(a_coeffs_df) >trh).sum(axis = "columns"))
display(a_coeffs_df)

CDK1_2      14
CDK4_6       8
EGFR        19
Estrogen     9
FGFR        10
PI3K        17
p53         26
TOP2A       10
Src          5
SMAD3        1
dtype: int64

,AARS,ABCB6,ABCC5,ABCF1,ABCF3,ABHD4,ABHD6,ABL1,ACAA1,ACAT2,...,ZMIZ1,ZMYM2,ZNF131,ZNF274,ZNF318,ZNF395,ZNF451,ZNF586,ZNF589,ZW10
CDK1_2,-1.643092e-05,6.510211e-06,9.942168e-07,0.000001,-2.483262e-06,-2.295695e-06,-0.000012,0.000010,-0.000001,-7.920432e-06,...,-0.000003,-0.000029,0.000005,0.000023,0.000012,-0.000004,-2.869161e-05,9.892916e-06,-0.000004,6.246373e-06
CDK4_6,-8.402072e-07,4.892862e-05,4.334128e-06,-0.000006,2.104527e-05,-1.699109e-05,0.000004,0.000014,0.000017,-7.566436e-06,...,-0.000020,0.000025,0.000007,-0.000010,-0.000012,-0.000020,1.177886e-05,1.309362e-05,-0.000016,-6.777186e-06
EGFR,-4.227527e-07,-2.987467e-05,-1.059733e-06,-0.000007,-6.303519e-06,-3.427611e-04,0.000029,-0.000026,0.000019,-2.671268e-02,...,0.000014,-0.000033,-0.000013,-0.000009,0.000012,0.000207,2.571255e-05,-2.500697e-05,-0.000008,9.904667e-06
Estrogen,-9.684646e-06,2.189894e-05,1.383697e-05,-0.000006,-1.257269e-05,-1.460977e-05,0.000007,-0.000009,-0.000036,-2.740553e-01,...,0.000013,0.000007,0.000006,-0.000009,-0.000030,0.000010,2.438238e-05,1.849796e-06,-0.000020,5.898869e-08
FGFR,-8.694749e-04,-3.379244e-07,-8.510971e-06,0.000007,2.333429e-05,-1.661751e-05,0.000009,0.000006,-0.000005,-1.207746e-05,...,0.000013,0.000017,-0.000019,0.000005,0.000002,0.000012,3.119531e-07,2.968841e-05,0.000011,1.340103e-05
PI3K,7.960737e-06,2.798068e-05,3.292835e-06,-0.000026,-6.885931e-06,5.641670e-07,-0.000019,-0.000003,0.000009,-1.297667e-05,...,0.000020,-0.000012,0.000020,-0.000016,0.000007,-0.000005,8.174900e-06,1.364927e-06,0.000012,-1.323413e-05
p53,-3.480310e-06,-1.567992e-06,2.267678e-05,0.000013,3.951325e-06,8.930460e-06,0.252449,0.000005,-0.000003,8.044327e-07,...,-0.000003,-0.000053,-0.000012,0.000015,0.000052,-0.000005,1.222270e-05,-1.663397e-05,0.000057,2.931894e-05
TOP2A,-3.252472e-06,1.020544e-05,-2.740051e-06,0.000005,-3.257597e-06,2.052726e-05,-0.000006,0.000003,-0.000003,7.518680e-06,...,0.000001,0.000010,0.000044,-0.000017,0.000019,0.000044,5.831384e-05,2.021263e-05,-0.000004,-6.151163e-05
Src,-1.121651e-05,1.092589e-05,4.783817e-06,0.000004,2.371011e-06,6.166803e-07,0.000031,0.000030,-0.000012,-5.689267e-05,...,0.000031,0.000024,-0.000006,0.000012,-0.000009,-0.000047,3.144051e-06,-1.001071e-05,0.000013,-1.073554e-05
SMAD3,-1.128676e-05,1.288040e-05,3.286412e-06,0.000018,-8.607800e-07,-2.539067e-05,0.000008,-0.000001,-0.000011,-1.904833e-05,...,0.000019,-0.000009,-0.000003,-0.000005,-0.000002,-0.000008,2.307231e-05,4.298863e-07,-0.000027,-3.218910e-05


In [9]:
#pathway_activity = a_coeffs @ x
#pathway_activity.shape

In [10]:
R_global = bmra_prep.pathway_activity.calc_global_response_from_pathway_activity(
    bmra_prep.pathway_activity.calc_pathway_activity(x,a_coeffs),
    modules,
    L1000_df.columns
)
R_global_df = R_global.dataframe
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD_V11,SMAD_V12,SMAD_V13,SMAD_V14,SMAD_V15,SMAD_V16,SMAD_V17,SMAD_V18,SMAD_V19,SMAD_V20
CDK1_2,-0.662092,-0.562302,0.089557,-0.075718,0.091398,-0.283312,0.044759,-0.305252,0.121154,0.069507,...,-0.489712,-0.443665,-0.405794,-0.423196,-0.391057,-0.407968,-0.399339,-0.444902,-0.441755,-0.480918
CDK4_6,-0.091145,-0.210863,-0.265789,-0.141281,-0.164233,-0.102887,-0.017154,-0.028890,-0.218126,-0.010608,...,0.198240,0.189154,0.177541,0.140912,0.138937,0.150573,0.172038,0.164713,0.190026,0.163245
EGFR,0.598071,0.498142,0.229191,0.371131,0.486361,0.114017,-0.415735,0.262344,0.287176,-0.047462,...,-1.229252,-0.823788,-0.809354,-0.709012,-0.679386,-0.694898,-0.746822,-0.902139,-0.881962,-0.936183
Estrogen,-0.128514,-0.215414,-0.214074,-0.411706,-0.954271,-0.307665,-0.087013,-0.244789,-0.168903,-0.042308,...,-0.294327,-0.263681,-0.237360,-0.261187,-0.242259,-0.250349,-0.228426,-0.274362,-0.258240,-0.301815
FGFR,-0.110193,-0.177174,-0.089425,0.051445,-0.028385,-0.406789,-0.032848,-0.028557,-0.062093,-0.300603,...,-0.857715,-0.702922,-0.641226,-0.641177,-0.571545,-0.620097,-0.627532,-0.714184,-0.719965,-0.795380
PI3K,-1.937702,-1.732367,-1.504565,-1.306942,-0.689689,-0.194951,-0.027226,-0.491595,-1.184232,-0.217054,...,-0.153300,-0.110103,-0.079957,-0.095071,-0.073442,-0.112816,-0.095638,-0.112380,-0.123251,-0.170377
p53,-0.268033,-0.271616,-0.180138,-0.401122,0.036073,-1.628591,-1.477271,-0.119102,-0.094001,-1.329931,...,0.058651,0.035221,0.047180,0.049062,0.020268,0.054278,0.073622,0.034066,0.074485,0.048759
TOP2A,-0.230056,0.071974,-0.235027,-0.174902,-0.127701,0.049738,0.068622,-2.000219,-0.207593,-0.291858,...,0.121443,0.106089,0.098170,0.082966,0.077505,0.092792,0.098971,0.098279,0.110489,0.110408
Src,-0.930860,-1.680300,0.522878,-1.206122,0.571361,-1.128651,0.495686,0.394117,0.396950,0.447102,...,-0.128221,-0.154486,-0.098596,-0.128326,-0.056932,-0.118359,-0.099334,-0.114242,-0.156242,-0.167021
SMAD3,0.017557,-0.097063,0.026413,-0.284454,0.048504,0.068257,0.021905,0.115999,0.118487,0.051918,...,-0.704586,-0.648577,-0.621137,-0.642726,-0.617305,-0.624631,-0.614176,-0.662227,-0.650816,-0.690022


In [11]:
R_global_df.to_csv(os.path.join(out_dir, "R_global_annotated.csv"))
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD_V11,SMAD_V12,SMAD_V13,SMAD_V14,SMAD_V15,SMAD_V16,SMAD_V17,SMAD_V18,SMAD_V19,SMAD_V20
CDK1_2,-0.662092,-0.562302,0.089557,-0.075718,0.091398,-0.283312,0.044759,-0.305252,0.121154,0.069507,...,-0.489712,-0.443665,-0.405794,-0.423196,-0.391057,-0.407968,-0.399339,-0.444902,-0.441755,-0.480918
CDK4_6,-0.091145,-0.210863,-0.265789,-0.141281,-0.164233,-0.102887,-0.017154,-0.028890,-0.218126,-0.010608,...,0.198240,0.189154,0.177541,0.140912,0.138937,0.150573,0.172038,0.164713,0.190026,0.163245
EGFR,0.598071,0.498142,0.229191,0.371131,0.486361,0.114017,-0.415735,0.262344,0.287176,-0.047462,...,-1.229252,-0.823788,-0.809354,-0.709012,-0.679386,-0.694898,-0.746822,-0.902139,-0.881962,-0.936183
Estrogen,-0.128514,-0.215414,-0.214074,-0.411706,-0.954271,-0.307665,-0.087013,-0.244789,-0.168903,-0.042308,...,-0.294327,-0.263681,-0.237360,-0.261187,-0.242259,-0.250349,-0.228426,-0.274362,-0.258240,-0.301815
FGFR,-0.110193,-0.177174,-0.089425,0.051445,-0.028385,-0.406789,-0.032848,-0.028557,-0.062093,-0.300603,...,-0.857715,-0.702922,-0.641226,-0.641177,-0.571545,-0.620097,-0.627532,-0.714184,-0.719965,-0.795380
PI3K,-1.937702,-1.732367,-1.504565,-1.306942,-0.689689,-0.194951,-0.027226,-0.491595,-1.184232,-0.217054,...,-0.153300,-0.110103,-0.079957,-0.095071,-0.073442,-0.112816,-0.095638,-0.112380,-0.123251,-0.170377
p53,-0.268033,-0.271616,-0.180138,-0.401122,0.036073,-1.628591,-1.477271,-0.119102,-0.094001,-1.329931,...,0.058651,0.035221,0.047180,0.049062,0.020268,0.054278,0.073622,0.034066,0.074485,0.048759
TOP2A,-0.230056,0.071974,-0.235027,-0.174902,-0.127701,0.049738,0.068622,-2.000219,-0.207593,-0.291858,...,0.121443,0.106089,0.098170,0.082966,0.077505,0.092792,0.098971,0.098279,0.110489,0.110408
Src,-0.930860,-1.680300,0.522878,-1.206122,0.571361,-1.128651,0.495686,0.394117,0.396950,0.447102,...,-0.128221,-0.154486,-0.098596,-0.128326,-0.056932,-0.118359,-0.099334,-0.114242,-0.156242,-0.167021
SMAD3,0.017557,-0.097063,0.026413,-0.284454,0.048504,0.068257,0.021905,0.115999,0.118487,0.051918,...,-0.704586,-0.648577,-0.621137,-0.642726,-0.617305,-0.624631,-0.614176,-0.662227,-0.650816,-0.690022
